# FTIR ML Solver v3 — Colab Training

## Before running
1. **Runtime → Change runtime type → GPU (T4)**
2. Run cells top to bottom

> ⏱ ~2 min/epoch on T4 with 10 000 synthetic + 557 reference samples.

In [ ]:
# Cell 1 — Clone repo and install
# The -c credential.helper='' flag disables credential lookup inline,
# which is required because Colab sets GIT_TERMINAL_PROMPT=0 globally.
!git -c credential.helper='' clone https://github.com/DrSuppe/FTIR_absorbtion_ML.git /content/ftir
import os; os.chdir('/content/ftir')
print('Working directory:', os.getcwd())
!pip install -e . -q

In [ ]:
# Cell 2 — Upload reference spectra (binary files not stored in the repo)
#
# On your Mac, zip them first:
#   cd /path/to/FTIR_ML_fitting
#   zip -r spc_files.zip reference_spectra/spc_files/
#
# Then run this cell and upload spc_files.zip when prompted.
import os
from google.colab import files

uploaded = files.upload()  # pick spc_files.zip
for fname in uploaded:
    if fname.endswith('.zip'):
        os.makedirs('reference_spectra', exist_ok=True)
        os.system(f'unzip -q "{fname}" -d reference_spectra/')
        print(f'Unzipped {fname}')

# Rebuild manifest from the newly uploaded SPC files
from ftir_analysis.manifesting import build_manifest
m = build_manifest()
print(f'Manifest ready: {len(m)} spectra, {m.species.nunique()} species')

In [ ]:
# Cell 3 — Quick smoke test (~10 min on T4)
!python3 train.py \
    --n-synthetic 5000 \
    --epochs 10 \
    --batch-size 64 \
    --log-every 2

In [ ]:
# Cell 4 — Full training run (T4, ~1-2 hrs)
# Uncomment and run instead of Cell 3 when ready.
# !python3 train.py \
#     --n-synthetic 50000 \
#     --epochs 100 \
#     --batch-size 128 \
#     --lr 3e-4 \
#     --warmup-epochs 5 \
#     --log-every 5

In [ ]:
# Cell 5 — [Optional] Mount Google Drive to persist checkpoints across sessions
# from google.colab import drive
# drive.mount('/content/drive')
# Then add to your train.py call:
#   --checkpoint-dir /content/drive/MyDrive/ftir_checkpoints

In [ ]:
# Cell 6 — View the latest per-species MAE plot
from pathlib import Path
import matplotlib.pyplot as plt, matplotlib.image as mpimg

plots = sorted(Path('reports').glob('mae_per_species_*.png'))
if plots:
    img = mpimg.imread(plots[-1])
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'MAE — {plots[-1].name}')
    plt.show()
else:
    print('No MAE plots yet — run training first.')

In [ ]:
# Cell 7 — Download best checkpoint to your local machine
from google.colab import files
files.download('checkpoints/best_model.pt')